In [1]:
import pandas as pd
import dotenv
from datetime import datetime, timedelta, timezone

dotenv.load_dotenv()

file_path = '../../data/reddit_submissions/SuicideWatch_submissions.jsonl'

# Sanity check dataset

In [2]:
with open(file_path, 'r', encoding='utf-8') as f:
    for i in range(2):
        print(f.readline())

{"archived":true,"author":"S2S2S2S2S2","author_flair_background_color":null,"author_flair_css_class":null,"author_flair_richtext":[],"author_flair_text":null,"author_flair_text_color":null,"author_flair_type":"text","brand_safe":false,"can_gild":true,"contest_mode":false,"created_utc":1229436284,"distinguished":null,"domain":"crisiscentre.bc.ca","edited":false,"gilded":0,"hidden":false,"hide_score":false,"id":"7jtxk","is_crosspostable":false,"is_reddit_media_domain":false,"is_self":false,"is_video":false,"link_flair_css_class":null,"link_flair_richtext":[],"link_flair_text":null,"link_flair_text_color":"dark","link_flair_type":"text","locked":false,"media":null,"media_embed":{},"no_follow":false,"num_comments":1,"num_crossposts":0,"over_18":false,"parent_whitelist_status":"no_ads","permalink":"\/r\/SuicideWatch\/comments\/7jtxk\/tips_to_help_us_all_cope_in_during_the_holidays\/","retrieved_on":1522768118,"rte_mode":"markdown","score":15,"secure_media":null,"secure_media_embed":{},"self

# Loading dataset into memory

In [3]:
TIME_FRAME = 365*2

chunksize = 100_000  # tune for your RAM

cutoff_dt = datetime.now(timezone.utc) - timedelta(days=TIME_FRAME)
cutoff_ts = cutoff_dt.timestamp()

frames = []

for chunk in pd.read_json(file_path, lines=True, chunksize=chunksize):
    # keep only last 2 years
    chunk = chunk[chunk["created_utc"] >= cutoff_ts]

    # drop deleted/removed posts
    chunk = chunk[~chunk["selftext"].isin(["[deleted]", "[removed]"])]
    chunk = chunk[~chunk["author"].isin(["[deleted]", "[removed]"])]

    if len(chunk):
        frames.append(chunk)

time_filtered_df = pd.concat(frames, ignore_index=True)
filtered_df = time_filtered_df[
    [
        "selftext",
        "created_utc",
        "title",
        "id",
        "subreddit",
        "subreddit_id",
        "retrieved_on",
    ]
].dropna()

# add t3_ prefix to id and rename to post_id
# chromadb uses "id" to identify documents
filtered_df["post_id"] = "t3_" + filtered_df["id"].astype(str)
filtered_df = filtered_df.drop(columns=["id"])

print(len(filtered_df), f"rows in last {TIME_FRAME} days after filtering")

225802 rows in last 730 days after filtering


# Sanity Check on `selftext` length

In [4]:
selftext_lengths = filtered_df["selftext"].fillna("").apply(lambda x: len(str(x).split()))
title_lengths = filtered_df["title"].fillna("").apply(lambda x: len(str(x).split()))

zero_len_df = filtered_df[selftext_lengths == 0]

print(f"Number of posts with 0-word selftext: {len(zero_len_df)}")

Number of posts with 0-word selftext: 4471


## Drop rows with less than X words

In [5]:
MIN_WORDS = 300

df_with_min_text = filtered_df[selftext_lengths > MIN_WORDS]
print(f"{len(df_with_min_text)} rows remain after dropping posts with {MIN_WORDS}-word selftext")

28889 rows remain after dropping posts with 300-word selftext


## Truncate posts on the last `.?!` punctuation, up to 400 words

In [6]:
import re

MAX_WORDS = 600
SEPARATOR = " | "

def truncate_to_sentence(text: str, max_words: int, min_words: int = 50) -> str:
    words = text.split()
    if len(words) <= max_words:
        return text

    truncated = ' '.join(words[:max_words])
    match = re.search(r'^(.*[.!?])[^.!?]*$', truncated, re.DOTALL)
    if match:
        result = match.group(1).strip()
        if len(result.split()) >= min_words:
            return result

    return truncated.strip()

def truncate_row(row) -> str:
    title = str(row["title"]) if pd.notna(row["title"]) else ""
    selftext = str(row["selftext"]) if pd.notna(row["selftext"]) else ""
    
    title_words = len(title.split())
    selftext_budget = max(0, MAX_WORDS - title_words)
    
    return truncate_to_sentence(selftext, selftext_budget)

original_word_counts = (
    df_with_min_text["title"].fillna("").apply(lambda x: len(x.split())) +
    df_with_min_text["selftext"].fillna("").apply(lambda x: len(x.split()))
)
n_truncated = (original_word_counts > MAX_WORDS).sum()

df_with_min_max_text = df_with_min_text.copy()
df_with_min_max_text["selftext"] = df_with_min_text.apply(truncate_row, axis=1)

print(f"Capped at {MAX_WORDS} words, truncated at sentence boundary.")
print(f"Rows affected: {n_truncated}")

if n_truncated > 0:
    example_idx = original_word_counts[original_word_counts > MAX_WORDS].index[0]
    row = df_with_min_max_text.loc[example_idx]
    doc = row["title"] + SEPARATOR + row["selftext"]
    total_words = len(doc.split())
    print(f"\nExample (post_id={row['post_id']}, original words={int(original_word_counts.loc[example_idx])}):")
    print(f"  Words now: {total_words}")
    print(f"  Title: {row['title'][:80]}")
    print(f"  Selftext ending: ...{row['selftext'][-300:]}")

Capped at 600 words, truncated at sentence boundary.
Rows affected: 6904

Example (post_id=t3_1db52g1, original words=665):
  Words now: 580
  Title: Does sobriety ever become enjoyable?
  Selftext ending: ...imately everything, and not one thing has helped me at all, except for drugs - which are also the very thing that was destroying me. I would go back to using but the only way I could do that is by lying to my family, my boyfriend, etc. which I cannot afford to do. I refuse to lie to the ones I love.


## Clean markdown formatting from selftext

In [7]:
# Audit: how much markdown actually exists in the selftext?
checks = {
    "bold/italic (**text** or *text*)": r'\*{1,2}.+?\*{1,2}',
    "blockquote (> at line start)":     r'(?m)^>',
    "header (# at line start)":         r'(?m)^#{1,6}\s',
    "strikethrough (~~text~~)":         r'~~.+?~~',
    "link ([text](url))":               r'\[[^\]]+\]\([^\)]+\)',
    "bare URL (http...)":               r'http\S+',
}

total = len(df_with_min_max_text)
print(f"Total posts: {total:,}\n")
for label, pattern in checks.items():
    n = df_with_min_max_text["selftext"].str.contains(pattern, regex=True, na=False).sum()
    print(f"  {label:<45} {n:>6,}  ({n/total:.1%})")

# Show one real example for each pattern that actually has hits
print("\n── Sample posts with markdown ──────────────────────────────")
for label, pattern in checks.items():
    mask = df_with_min_max_text["selftext"].str.contains(pattern, regex=True, na=False)
    if mask.any():
        example = df_with_min_max_text.loc[mask.idxmax(), "selftext"]
        print(f"\n[{label}]")
        print(example[:300])

Total posts: 28,889

  bold/italic (**text** or *text*)               1,541  (5.3%)
  blockquote (> at line start)                      20  (0.1%)
  header (# at line start)                          23  (0.1%)
  strikethrough (~~text~~)                          11  (0.0%)
  link ([text](url))                                46  (0.2%)
  bare URL (http...)                                91  (0.3%)

── Sample posts with markdown ──────────────────────────────

[bold/italic (**text** or *text*)]
I'm a 32 year old hermit who's been completely isolated from everything for well over half of my entire existence on this planet. I've never had friends. I've never dated, or had a significant other. Any attempts I've made to form connections with other people, which can only happen online due to my

[blockquote (> at line start)]
>To 16year old me,

>From today's me



**Subject - A last farewell...**

Hey! I received your letter this morning and I am writing this after reading your demands from m

In [8]:
import re

def clean_markdown(text):
    text = re.sub(r'\*{1,2}(.+?)\*{1,2}', r'\1', text)         # bold/italic
    text = re.sub(r'~~(.+?)~~', r'\1', text)                    # strikethrough
    text = re.sub(r'^#{1,6}\s+', '', text, flags=re.MULTILINE)  # headers
    text = re.sub(r'^>\s+', '', text, flags=re.MULTILINE)       # blockquotes
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)       # links → text
    text = re.sub(r'http\S+', '', text)                         # bare URLs
    text = re.sub(r'\n{3,}', '\n\n', text)                      # excess newlines
    return text.strip()

# Find a post that actually contains markdown to make the diff meaningful
markdown_indicators = r'[*#>~\[\]]|http'
sample_idx = df_with_min_max_text["selftext"].str.contains(markdown_indicators, regex=True).idxmax()

original = df_with_min_max_text.loc[sample_idx, "selftext"]
cleaned = clean_markdown(original)

print("── BEFORE ──────────────────────────────────────────────────")
print(original[:600])
print("\n── AFTER ───────────────────────────────────────────────────")
print(cleaned[:600])

# Apply to full dataframe
df_with_clean_markdown = df_with_min_max_text.copy()
df_with_clean_markdown["selftext"] = df_with_min_max_text["selftext"].apply(clean_markdown)
print(f"\nMarkdown cleaned across {len(df_with_clean_markdown):,} rows.")

── BEFORE ──────────────────────────────────────────────────
I'm 18M and I live in small city in india. I wont go in a full detail about my life but from childhood I've faced bullying alot and my family's toxic behavior too. In my family i have my father, my mother, 2 elder sisters, my grandmother & 2 dogs. For some little context my father Is abusive towards my mother and my mother has also devloped an extreme toxic behavior in a recent years. In past when I was 2 years old my best friend and my grandfather passed away suddenly, a girl I used to love left me for someone guy she told me not to worry about and at last a girl who is very characterless and

── AFTER ───────────────────────────────────────────────────
I'm 18M and I live in small city in india. I wont go in a full detail about my life but from childhood I've faced bullying alot and my family's toxic behavior too. In my family i have my father, my mother, 2 elder sisters, my grandmother & 2 dogs. For some little context my f

In [9]:
# Audit: how much markdown actually exists in the selftext?
checks = {
    "bold/italic (**text** or *text*)": r'\*{1,2}.+?\*{1,2}',
    "blockquote (> at line start)":     r'(?m)^>',
    "header (# at line start)":         r'(?m)^#{1,6}\s',
    "strikethrough (~~text~~)":         r'~~.+?~~',
    "link ([text](url))":               r'\[[^\]]+\]\([^\)]+\)',
    "bare URL (http...)":               r'http\S+',
}

total = len(df_with_clean_markdown)
print(f"Total posts: {total:,}\n")
for label, pattern in checks.items():
    n = df_with_clean_markdown["selftext"].str.contains(pattern, regex=True, na=False).sum()
    print(f"  {label:<45} {n:>6,}  ({n/total:.1%})")

# Show one real example for each pattern that actually has hits
print("\n── Sample posts with markdown ──────────────────────────────")
for label, pattern in checks.items():
    mask = df_with_clean_markdown["selftext"].str.contains(pattern, regex=True, na=False)
    if mask.any():
        example = df_with_clean_markdown.loc[mask.idxmax(), "selftext"]
        print(f"\n[{label}]")
        print(example[:300])

Total posts: 28,889

  bold/italic (**text** or *text*)                  91  (0.3%)
  blockquote (> at line start)                      19  (0.1%)
  header (# at line start)                           0  (0.0%)
  strikethrough (~~text~~)                           0  (0.0%)
  link ([text](url))                                 0  (0.0%)
  bare URL (http...)                                 0  (0.0%)

── Sample posts with markdown ──────────────────────────────

[bold/italic (**text** or *text*)]
I've posted here before. 

I've had nights where I knew I wouldn't make it back, even though I'm like a cat  

I'm the person that can rage with a crew, and they all know Sed gets them back to the car and home safely. 

Last week I parted ways with a friend I made, telling them that I was looking ou

[blockquote (> at line start)]
>To 16year old me,

>From today's me

Subject - A last farewell...

Hey! I received your letter this morning and I am writing this after reading your demands from me.

I 

In [11]:
import re

# Drop posts that still contain markdown after cleaning
# re.MULTILINE needed so ^ anchors match line starts, not just string start
residual_markdown = r'\*{1,2}.+?\*{1,2}|^>|^#{1,6}\s|~~.+?~~|\[[^\]]+\]\([^\)]+\)|http\S+'

before = len(df_with_clean_markdown)
df_with_clean_markdown = df_with_clean_markdown[
    ~df_with_clean_markdown["selftext"].str.contains(
        residual_markdown, regex=True, na=False, flags=re.MULTILINE
    )
].reset_index(drop=True)

dropped = before - len(df_with_clean_markdown)
print(f"Dropped {dropped:,} posts with residual markdown ({dropped/before:.1%})")
print(f"Remaining: {len(df_with_clean_markdown):,}")

Dropped 110 posts with residual markdown (0.4%)
Remaining: 28,779


## Filter non-English posts

In [12]:
from langdetect import detect, LangDetectException
from tqdm import tqdm

tqdm.pandas()

def is_english(text: str) -> bool:
    try:
        return detect(str(text)) == 'en'
    except LangDetectException:
        return False

english_mask = df_with_clean_markdown["selftext"].progress_apply(is_english)
english_filter_df = df_with_clean_markdown[english_mask].reset_index(drop=True)

n_removed = len(df_with_clean_markdown) - len(english_filter_df)
print(f"Removed {n_removed:,} non-English posts ({n_removed / len(df_with_clean_markdown):.1%})")
print(f"Remaining: {len(english_filter_df):,}")

100%|██████████| 28779/28779 [01:51<00:00, 258.92it/s]


Removed 94 non-English posts (0.3%)
Remaining: 28,685


# Persona Profile
- Age: 14-19
- Gender: Girl
- Issues:
    - Bullying
    - Self Harm
    - Sexual Assault
    - Suicidal

# Build Regex search

In [13]:
import re

# ── 1. Age signals ────────────────────────────────────────────────────────────
AGE_PATTERNS = re.compile(
    r'\b(i[\'m\s]+\d{1,2}|i am \d{1,2}|'
    r'\d{1,2}f\b|\d{1,2} f\b|'           # "16f", "15 f"
    r'(14|15|16|17|18|19)\s*(year|yo|y/o)|'
    r'freshman|sophomore|junior|senior|'
    r'high school|highschool|grade \d+)\b',
    re.IGNORECASE
)

# ── 2. Female gender signals ───────────────────────────────────────────────────
GENDER_PATTERNS = re.compile(
    r'\b(she/her|i\'m a girl|i am a girl|'
    r'as a girl|as a woman|'
    r'\d{1,2}f\b|\d{1,2} f\b|'
    r'my boyfriend|my mom told me|'
    r'female|girl|woman)\b',
    re.IGNORECASE
)

# ── 3. Self-harm signals ───────────────────────────────────────────────────────
SELFHARM_PATTERNS = re.compile(
    r'\b(cut(ting)? myself|self.harm|self harm|'
    r'cutting|burn(ing)? myself|'
    r'hurt(ing)? myself|scars|razor|blade)\b',
    re.IGNORECASE
)

# ── 4. Bullying signals ────────────────────────────────────────────────────────
BULLYING_PATTERNS = re.compile(
    r'\b(bully|bullied|bullying|'
    r'picked on|make fun of me|'
    r'no friends|eat alone|'
    r'everyone hates me|they laugh at me|hates me|nobody likes me|'
    r'excluded|left out|ostracised|'
    r'school is|hate school|'
    r'group chat|screenshots|'
    r'cyber.?bully)\b',
    re.IGNORECASE
)

# ── 5. Sexual assault signals ─────────────────────────────────────────────────
SA_PATTERNS = re.compile(
    r'\b(touched me|touched without|'
    r'sexual(ly)?|harass(ed|ment)|'
    r'assault(ed)?|groped|'
    r'made me feel uncomfortable|'
    r'older (guy|man|boy)|'
    r'didn\'t consent|violated|'
    r'unwanted|he wouldn\'t stop)\b'
    r'|(?-i:\bSA\b)',
    re.IGNORECASE
)

# ── 6. Suicidal ideation signals ──────────────────────────────────────────────
SUICIDAL_PATTERNS = re.compile(
    r'\b(want(ing)? to die|wanna die|'
    r'kill(ing)? myself|end my life|'
    r'no reason to live|not worth living|'
    r'rather be dead|better off dead|'
    r'don\'t want to (be here|live|exist)|'
    r'suicidal|suicide|'
    r'take my (own )?life|'
    r'end it all|ending it all|'
    r'can\'t go on|don\'t want to go on)\b',
    re.IGNORECASE
)

In [14]:
PATTERNS = {
    "age":      AGE_PATTERNS,
    "female":   GENDER_PATTERNS,
    "selfharm": SELFHARM_PATTERNS,
    "bullying": BULLYING_PATTERNS,
    "sa":       SA_PATTERNS,
    "suicidal": SUICIDAL_PATTERNS,
}

def get_tags(text: str) -> list[str]:
    return [tag for tag, pat in PATTERNS.items() if pat.search(text)]

# Apply to title + selftext combined so title signals aren't missed
combined = english_filter_df["title"].fillna("") + " " + english_filter_df["selftext"].fillna("")

tagged_df = english_filter_df.copy()
tagged_df["tags"]      = combined.apply(get_tags)
tagged_df["tag_count"] = tagged_df["tags"].apply(len)

ranked_df = tagged_df.sort_values("tag_count", ascending=False).reset_index(drop=True)

print(f"Total posts: {len(ranked_df):,}\n")
print("Tag count distribution:")
print(ranked_df["tag_count"].value_counts().sort_index(ascending=False).to_string())
print(f"\nTop 5 posts:")
print(ranked_df[["post_id", "tag_count", "tags", "title"]].head())

Total posts: 28,685

Tag count distribution:
tag_count
6       22
5      274
4     1287
3     4229
2     8274
1    10539
0     4060

Top 5 posts:
      post_id  tag_count                                             tags  \
0  t3_1fvcrw3          6  [age, female, selfharm, bullying, sa, suicidal]   
1  t3_1frsnkc          6  [age, female, selfharm, bullying, sa, suicidal]   
2  t3_1djbnvd          6  [age, female, selfharm, bullying, sa, suicidal]   
3  t3_1hwywkd          6  [age, female, selfharm, bullying, sa, suicidal]   
4  t3_1fuwjws          6  [age, female, selfharm, bullying, sa, suicidal]   

                                               title  
0  I hate my mom so fucking much. She is the root...  
1                      Thinking about ending it all.  
2  I hate living and have always felt like I was ...  
3                                      Man I’m done   
4            I’m actually losing it. Please help me   


In [ ]:
RELEVANT_TAG_COUNT = 4

relevant_tagged_df = ranked_df[ranked_df["tag_count"] >= RELEVANT_TAG_COUNT].reset_index(drop=True)

print(f"Posts with {RELEVANT_TAG_COUNT}+ tags: {len(relevant_tagged_df):,}")
print(f"Dropped: {len(ranked_df) - len(relevant_tagged_df):,}")
print(f"\nTag count distribution:")
print(relevant_tagged_df["tag_count"].value_counts().sort_index(ascending=False).to_string())

Posts with 4+ tags: 1,583
Dropped: 27,102

Tag count distribution:
tag_count
6      22
5     274
4    1287


In [ ]:
relevant_tagged_df.to_csv("/hannah_semantic_exploration_df.csv")

In [20]:
import tiktoken

# Input token pricing per 1M tokens (OpenAI standard tier, June 2026)
# Grouped by generation for readability
MODELS = {
    # ── Latest ────────────────────────────────────────────────
    "gpt-4.1":              ("cl100k_base",  2.00),
    "gpt-4.1-mini":         ("cl100k_base",  0.40),
    "gpt-4.1-nano":         ("cl100k_base",  0.10),
    # ── gpt-4o family ─────────────────────────────────────────
    "gpt-4o":               ("o200k_base",   2.50),
    "gpt-4o-mini":          ("o200k_base",   0.15),
    # ── gpt-4 legacy ──────────────────────────────────────────
    "gpt-4-turbo":          ("cl100k_base", 10.00),
    "gpt-4":                ("cl100k_base", 30.00),
    "gpt-4-32k":            ("cl100k_base", 60.00),
    # ── gpt-3.5 ───────────────────────────────────────────────
    "gpt-3.5-turbo":        ("cl100k_base",  0.50),
    # ── Embeddings ────────────────────────────────────────────
    "text-embedding-3-small": ("cl100k_base", 0.02),
    "text-embedding-3-large": ("cl100k_base", 0.13),
    "text-embedding-ada-002": ("cl100k_base", 0.10),
}

# Count tokens once per encoding (cl100k vs o200k differ slightly)
def count_tokens(df, models=MODELS):
    titles    = df["title"].fillna("")
    selftexts = df["selftext"].fillna("")

    token_counts = {}
    for enc_name in set(enc for enc, _ in models.values()):
        enc = tiktoken.get_encoding(enc_name)
        total = 0
        for i, (title, selftext) in enumerate(zip(titles, selftexts), 1):
            total += len(enc.encode(f"{title} | {selftext}"))
            if i % 500 == 0:
                print(f"  [{enc_name}] {i:>5} / {len(df):,} rows")
        token_counts[enc_name] = total
        print(f"  [{enc_name}] total tokens: {total:,}")

    print(f"\n{'Model':<30} {'Encoding':<15} {'$/1M in':>9}   {'Est. cost':>10}")
    print("─" * 70)
    last_group = None
    for model, (enc_name, cost_per_million) in models.items():
        group = model.split("-")[0] + "-" + model.split("-")[1] if "-" in model else model
        if last_group and group != last_group:
            print()
        last_group = group
        total = token_counts[enc_name]
        cost  = total / 1_000_000 * cost_per_million
        print(f"{model:<30} {enc_name:<15} ${cost_per_million:>7.2f}   ${cost:>9.4f}")

    return token_counts

token_counts = count_tokens(relevant_tagged_df)

  [cl100k_base]   500 / 1,583 rows
  [cl100k_base]  1000 / 1,583 rows
  [cl100k_base]  1500 / 1,583 rows
  [cl100k_base] total tokens: 922,706
  [o200k_base]   500 / 1,583 rows
  [o200k_base]  1000 / 1,583 rows
  [o200k_base]  1500 / 1,583 rows
  [o200k_base] total tokens: 907,109

Model                          Encoding          $/1M in    Est. cost
──────────────────────────────────────────────────────────────────────
gpt-4.1                        cl100k_base     $   2.00   $   1.8454
gpt-4.1-mini                   cl100k_base     $   0.40   $   0.3691
gpt-4.1-nano                   cl100k_base     $   0.10   $   0.0923

gpt-4o                         o200k_base      $   2.50   $   2.2678
gpt-4o-mini                    o200k_base      $   0.15   $   0.1361

gpt-4-turbo                    cl100k_base     $  10.00   $   9.2271
gpt-4                          cl100k_base     $  30.00   $  27.6812
gpt-4-32k                      cl100k_base     $  60.00   $  55.3624

gpt-3.5-turbo        

In [ ]:
MIN_TAG_COUNT = 5

hannah_raw_posts = ranked_df[ranked_df["tag_count"] == 6].reset_index(drop=True)

print(f"{len(hannah_raw_posts)} posts with all 6 tags\n")
for i, row in hannah_raw_posts.iterrows():
    print(f"{'='*70}")
    print(f"[{i+1}/{len(hannah_raw_posts)}]  {row['post_id']}")
    print(f"Title: {row['title']}")
    print(f"Tags:  {row['tags']}")
    print()
    print(row['selftext'])
    print()

In [ ]:
hannah_raw_posts.to_csv("../../data/hannah_raw_posts.csv")